In [1]:
import numpy as np
import gfootball.env as football_env

env = football_env.create_environment(
    env_name='5_vs_5',
    representation='raw',
    number_of_left_players_agent_controls=4,
    number_of_right_players_agent_controls=0,
    render=False,
)
obs = env.reset()
print("Environment created. Number of controlled agents:", len(obs))

Environment created. Number of controlled agents: 4


In [2]:
print("=" * 70)
print("FULL RAW OBSERVATION DUMP -- agent 0")
print("=" * 70)
for key in obs[0].keys():
    val = obs[0][key]
    val_arr = np.array(val)
    print(f"\nKey: '{key}'")
    print(f"  Type:  {type(val)}")
    print(f"  Shape: {val_arr.shape if val_arr.shape else 'scalar'}")
    print(f"  Value: {val}")

FULL RAW OBSERVATION DUMP -- agent 0

Key: 'steps_left'
  Type:  <class 'int'>
  Shape: scalar
  Value: 3001

Key: 'right_team_tired_factor'
  Type:  <class 'numpy.ndarray'>
  Shape: (5,)
  Value: [0. 0. 0. 0. 0.]

Key: 'ball_rotation'
  Type:  <class 'numpy.ndarray'>
  Shape: (3,)
  Value: [ 0. -0.  0.]

Key: 'ball'
  Type:  <class 'numpy.ndarray'>
  Shape: (3,)
  Value: [ 0.         -0.          0.11061639]

Key: 'right_team_yellow_card'
  Type:  <class 'numpy.ndarray'>
  Shape: (5,)
  Value: [False False False False False]

Key: 'ball_direction'
  Type:  <class 'numpy.ndarray'>
  Shape: (3,)
  Value: [-0.          0.          0.00616395]

Key: 'ball_owned_player'
  Type:  <class 'int'>
  Shape: scalar
  Value: -1

Key: 'score'
  Type:  <class 'list'>
  Shape: (2,)
  Value: [0, 0]

Key: 'game_mode'
  Type:  <class 'int'>
  Shape: scalar
  Value: 0

Key: 'left_team_tired_factor'
  Type:  <class 'numpy.ndarray'>
  Shape: (5,)
  Value: [0. 0. 0. 0. 0.]

Key: 'right_team_direction'
  Typ

In [3]:
print("left_team shape:       ", np.array(obs[0]['left_team']).shape)
print("left_team_roles:       ", obs[0]['left_team_roles'])
print("left_team_active:      ", obs[0]['left_team_active'])
print("right_team shape:      ", np.array(obs[0]['right_team']).shape)
print("right_team_active:     ", obs[0]['right_team_active'])
print("designated (player idx that is 'designated' by the game):", obs[0]['designated'])
print("active (currently-controlled player idx per controlled agent):")
for i in range(len(obs)):
    print(f"  agent {i}: active player index = {obs[i]['active']}")

n_active_left = int(np.sum(obs[0]['left_team_active']))
print(f"\nNumber of active left-team players: {n_active_left}  "
      f"(expect 5 for a 5v5 scenario)")
if n_active_left != 5:
    print("!! WARNING: active count != 5 -- your position slice [:5] may be wrong.")

left_team shape:        (5, 2)
left_team_roles:        [0 7 9 2 1]
left_team_active:       [ True  True  True  True  True]
right_team shape:       (5, 2)
right_team_active:      [ True  True  True  True  True]
designated (player idx that is 'designated' by the game): 2
active (currently-controlled player idx per controlled agent):
  agent 0: active player index = 1
  agent 1: active player index = 2
  agent 2: active player index = 3
  agent 3: active player index = 4

Number of active left-team players: 5  (expect 5 for a 5v5 scenario)


In [4]:
print("Ground-truth score field: obs[0]['score'] =", obs[0]['score'])
print("This is [left_score, right_score] directly from the engine.")
print("\nsteps_left:", obs[0]['steps_left'])

# Take a handful of random steps and track BOTH methods side by side
score_true_log = []
score_inferred = [0, 0]
for step in range(50):
    actions = env.action_space.sample()
    obs, reward, done, info = env.step(actions)
    step_reward = float(np.sum(reward)) if hasattr(reward, '__len__') else float(reward)
    if step_reward > 0:
        score_inferred[0] += 1
    elif step_reward < 0:
        score_inferred[1] += 1
    score_true_log.append(list(obs[0]['score']))
    if done:
        obs = env.reset()

print("\nTrue score field after 50 random steps:", score_true_log[-1])
print("Reward-sign-inferred score after 50 steps:", score_inferred)
print("\n(In 50 random steps a goal is unlikely either way -- the point is to")
print(" confirm obs[0]['score'] exists, updates correctly, and should be used")
print(" as ground truth instead of inferring from reward sign in future runs.)")

Ground-truth score field: obs[0]['score'] = [0, 0]
This is [left_score, right_score] directly from the engine.

steps_left: 3001

True score field after 50 random steps: [0, 0]
Reward-sign-inferred score after 50 steps: [0, 0]

(In 50 random steps a goal is unlikely either way -- the point is to
 confirm obs[0]['score'] exists, updates correctly, and should be used
 as ground truth instead of inferring from reward sign in future runs.)


In [5]:
print("ball_owned_team values seen over steps: -1=none, 0=left(us), 1=right")
possession_values_seen = set()
env.reset()
for step in range(200):
    actions = env.action_space.sample()
    obs, reward, done, info = env.step(actions)
    possession_values_seen.add(int(obs[0]['ball_owned_team']))
    if done:
        obs = env.reset()

print("Distinct ball_owned_team values observed in 200 steps:", possession_values_seen)
print("Expect to see -1 (none/contested) and possibly 0 and/or 1 depending on play")
print("\nball_owned_player (index of player who owns ball, -1 if none):", obs[0]['ball_owned_player'])

ball_owned_team values seen over steps: -1=none, 0=left(us), 1=right
Distinct ball_owned_team values observed in 200 steps: {1, -1}
Expect to see -1 (none/contested) and possibly 0 and/or 1 depending on play

ball_owned_player (index of player who owns ball, -1 if none): -1


In [6]:
ACTION_SPRINT = 13
ACTION_RELEASE_SPRINT = 15
SPRINT_STICKY_INDEX = 8  # previously confirmed

obs = env.reset()
before = obs[0]['sticky_actions'].copy() if hasattr(obs[0]['sticky_actions'], 'copy') else list(obs[0]['sticky_actions'])
actions = [ACTION_SPRINT, 0, 0, 0]
obs, _, _, _ = env.step(actions)
after = obs[0]['sticky_actions']

print("Before SPRINT:", before)
print("After SPRINT: ", after)
print(f"\nBit at SPRINT_STICKY_INDEX={SPRINT_STICKY_INDEX}: "
      f"{before[SPRINT_STICKY_INDEX]} -> {after[SPRINT_STICKY_INDEX]}")

if after[SPRINT_STICKY_INDEX] == 1 and before[SPRINT_STICKY_INDEX] == 0:
    print("PASS: index 8 still behaves as sprint bit.")
else:
    print("FAIL: regression -- index 8 no longer matches expected sprint behaviour. Investigate.")

Before SPRINT: [0 0 0 0 0 0 0 0 0 0]
After SPRINT:  [0 0 0 0 0 0 0 0 1 0]

Bit at SPRINT_STICKY_INDEX=8: 0 -> 1
PASS: index 8 still behaves as sprint bit.


In [7]:
env.reset()
all_positions = []

for step in range(1000):
    actions = env.action_space.sample()
    obs, reward, done, info = env.step(actions)
    all_positions.append(np.array(obs[0]['left_team']))
    all_positions.append(np.array(obs[0]['right_team']))
    if done:
        obs = env.reset()

all_positions = np.concatenate(all_positions, axis=0)  # (N, 2)

x_min, x_max = all_positions[:, 0].min(), all_positions[:, 0].max()
y_min, y_max = all_positions[:, 1].min(), all_positions[:, 1].max()

print(f"Observed x range: [{x_min:.4f}, {x_max:.4f}]  (assumed: [-1, 1])")
print(f"Observed y range: [{y_min:.4f}, {y_max:.4f}]  (assumed: [-0.42, 0.42])")

empirical_area = (x_max - x_min) * (y_max - y_min)
print(f"\nEmpirical bounding area from 1000 random steps: {empirical_area:.4f}")
print(f"Assumed GRF_PITCH_AREA constant currently in use: 1.68")
print("(Random policy in 1000 steps may not reach every pitch extreme -- treat")
print(" this as a lower bound check, not exact confirmation, but flag if wildly off.)")

Observed x range: [-1.0110, 1.0110]  (assumed: [-1, 1])
Observed y range: [-0.2996, 0.4350]  (assumed: [-0.42, 0.42])

Empirical bounding area from 1000 random steps: 1.4853
Assumed GRF_PITCH_AREA constant currently in use: 1.68
(Random policy in 1000 steps may not reach every pitch extreme -- treat
 this as a lower bound check, not exact confirmation, but flag if wildly off.)


In [8]:
print("Action space:", env.action_space)
print("Sample action:", env.action_space.sample())
print("Action space per agent - n:", env.action_space.nvec if hasattr(env.action_space, 'nvec') else env.action_space.n)

Action space: MultiDiscrete([19 19 19 19])
Sample action: [ 7 10 16  5]
Action space per agent - n: [19 19 19 19]


In [9]:
env.reset()
reward_log = []
score_log = []

for step in range(500):
    actions = env.action_space.sample()
    obs, reward, done, info = env.step(actions)
    reward_log.append(reward)
    score_log.append(list(obs[0]['score']))
    if done:
        obs = env.reset()

reward_arr = np.array(reward_log)
print("Reward array shape over 500 steps:", reward_arr.shape)
print("Reward dtype:", reward_arr.dtype)
print("Unique reward values seen:", np.unique(reward_arr))
print("\nFinal score after 500 random steps:", score_log[-1])

nonzero_reward_steps = np.where(np.any(reward_arr != 0, axis=1) if reward_arr.ndim > 1 else reward_arr != 0)[0]
print(f"Steps with non-zero reward: {len(nonzero_reward_steps)}")
if len(nonzero_reward_steps) > 0:
    print(f"First few: {nonzero_reward_steps[:5]}")
    for idx in nonzero_reward_steps[:3]:
        print(f"  step {idx}: reward={reward_log[idx]}, score at that step={score_log[idx]}")

Reward array shape over 500 steps: (500, 4)
Reward dtype: float32
Unique reward values seen: [0.]

Final score after 500 random steps: [0, 0]
Steps with non-zero reward: 0


In [10]:
env.reset()
game_modes_seen = set()
step_count = 0
done = False

while not done and step_count < 3500:  # safety cap above expected 3000
    actions = env.action_space.sample()
    obs, reward, done, info = env.step(actions)
    game_modes_seen.add(int(obs[0]['game_mode']))
    step_count += 1

print(f"Episode ended after {step_count} steps (requested max ~3000)")
print(f"steps_left at end: {obs[0]['steps_left']}")
print(f"Distinct game_mode values seen: {game_modes_seen}")
print("(0=Normal typically; others correspond to KickOff/GoalKick/FreeKick/Corner/ThrowIn/Penalty)")
print(f"Final score: {obs[0]['score']}")

Episode ended after 3001 steps (requested max ~3000)
steps_left at end: 0
Distinct game_mode values seen: {0, 2, 3, 5}
(0=Normal typically; others correspond to KickOff/GoalKick/FreeKick/Corner/ThrowIn/Penalty)
Final score: [0, 3]


In [11]:
print("=" * 70)
print("INTEGRATION WALKTHROUGH -- 10 steps, everything together")
print("=" * 70)

obs = env.reset()
for step in range(10):
    actions = env.action_space.sample()
    sprint_bits = [int(a['sticky_actions'][SPRINT_STICKY_INDEX]) for a in obs]
    positions = np.array(obs[0]['left_team'])[:5]
    possession = bool(obs[0]['ball_owned_team'] == 0)
    score = obs[0]['score']
    game_mode = obs[0]['game_mode']

    print(f"\nStep {step}:")
    print(f"  actions taken:      {actions}")
    print(f"  sprint bits/agent:  {sprint_bits}")
    print(f"  team positions[:2]: {positions[:2]}  (showing first 2 of 5)")
    print(f"  our possession:     {possession}")
    print(f"  score [L,R]:        {score}")
    print(f"  game_mode:          {game_mode}")

    obs, reward, done, info = env.step(actions)
    if done:
        obs = env.reset()

env.close()
print("\n[DONE] Integration walkthrough complete.")

INTEGRATION WALKTHROUGH -- 10 steps, everything together

Step 0:
  actions taken:      [10  2 13  9]
  sprint bits/agent:  [0, 0, 0, 0]
  team positions[:2]: [[-1.01102936 -0.        ]
 [ 0.          0.02032536]]  (showing first 2 of 5)
  our possession:     False
  score [L,R]:        [0, 0]
  game_mode:          0

Step 1:
  actions taken:      [ 7  8 16 14]
  sprint bits/agent:  [0, 0, 1, 0]
  team positions[:2]: [[-1.01102936e+00 -0.00000000e+00]
 [ 1.94497293e-10  1.98987499e-02]]  (showing first 2 of 5)
  our possession:     False
  score [L,R]:        [0, 0]
  game_mode:          0

Step 2:
  actions taken:      [ 9  5  3 11]
  sprint bits/agent:  [0, 0, 1, 0]
  team positions[:2]: [[-1.01102936e+00 -0.00000000e+00]
 [-2.96217360e-04  1.62396617e-02]]  (showing first 2 of 5)
  our possession:     False
  score [L,R]:        [0, 0]
  game_mode:          0

Step 3:
  actions taken:      [17  1  7  5]
  sprint bits/agent:  [0, 0, 1, 0]
  team positions[:2]: [[-1.00861216e+00  1.76

In [12]:
print("=" * 70)
print("CHECKING NOTEBOOK -- SUMMARY CHECKLIST")
print("=" * 70)
print("""
[ ] Cell 3: n_active_left == 5 confirmed?
[ ] Cell 4: obs[0]['score'] exists and updates -- SWITCH randompolicy.ipynb
    to use this instead of reward-sign inference
[ ] Cell 5: possession values include -1 and at least one of {0,1} -- confirms
    ball_owned_team semantics match extract_possession()
[ ] Cell 6: sprint regression PASS
[ ] Cell 7: empirical pitch bounds roughly consistent with GRF_PITCH_AREA=1.68
    assumption -- if wildly different, recompute the constant
[ ] Cell 9: reward is sparse (mostly 0, occasional +-1 on goals) as expected
    for 'scoring' reward type
[ ] Cell 10: episode naturally terminates near 3000 steps; game_mode values
    match expected set

Any unchecked box above = fix before trusting the 500-episode baseline run.
""")

CHECKING NOTEBOOK -- SUMMARY CHECKLIST

[ ] Cell 3: n_active_left == 5 confirmed?
[ ] Cell 4: obs[0]['score'] exists and updates -- SWITCH randompolicy.ipynb
    to use this instead of reward-sign inference
[ ] Cell 5: possession values include -1 and at least one of {0,1} -- confirms
    ball_owned_team semantics match extract_possession()
[ ] Cell 6: sprint regression PASS
[ ] Cell 7: empirical pitch bounds roughly consistent with GRF_PITCH_AREA=1.68
    assumption -- if wildly different, recompute the constant
[ ] Cell 9: reward is sparse (mostly 0, occasional +-1 on goals) as expected
    for 'scoring' reward type
[ ] Cell 10: episode naturally terminates near 3000 steps; game_mode values
    match expected set

Any unchecked box above = fix before trusting the 500-episode baseline run.

